# Global War & Conflict Impact (1950-2024) - EDA

Exploratory Data Analysis of a global conflicts dataset spanning 75 years. The dataset covers 3,000 conflicts across 15 countries with data on casualties, economic impact, military operations, geography, diplomacy, and outcomes.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12

## 1. Data Loading & Overview

In [ ]:
df = pd.read_csv('/kaggle/input/datasets/khushikyad001/global-war-and-conflict-impact-dataset-19502024/global_conflicts_dataset.csv')

# Derived columns
df['Total_Military_Deaths'] = df['Military_Deaths_A'] + df['Military_Deaths_B']
df['Total_Deaths'] = df['Total_Military_Deaths'] + df['Civilian_Deaths']
df['Decade'] = (df['Year'] // 10) * 10

print(f'Shape: {df.shape}')
print(f'Year range: {df["Year"].min()} - {df["Year"].max()}')
print(f'Countries: {sorted(df["Country_A"].unique())}')
print(f'\nMissing values:')
print(df.isnull().sum()[df.isnull().sum() > 0])
df.head()

In [ ]:
df.describe().round(2)

## 2. Conflict Timeline

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Total conflicts per year
yearly = df.groupby('Year').size()
axes[0].bar(yearly.index, yearly.values, color='steelblue', alpha=0.7)
axes[0].set_title('Number of Conflicts per Year (1950-2024)')
axes[0].set_ylabel('Count')
axes[0].axhline(yearly.mean(), color='red', linestyle='--', alpha=0.5, label=f'Mean: {yearly.mean():.1f}')
axes[0].legend()

# Conflicts by type over time (stacked area)
type_yearly = df.groupby(['Year', 'Conflict_Type']).size().unstack(fill_value=0)
type_colors = {'War': '#C0392B', 'Civil War': '#E74C3C', 'Proxy War': '#F39C12', 
               'Cold Conflict': '#3498DB', 'Skirmish': '#95A5A6'}
type_yearly.plot.area(ax=axes[1], stacked=True, alpha=0.7, 
                       color=[type_colors.get(c, '#888') for c in type_yearly.columns])
axes[1].set_title('Conflict Types Over Time')
axes[1].set_ylabel('Count')
axes[1].set_xlabel('Year')
axes[1].legend(title='Type', bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
plt.show()

In [ ]:
# Conflict type breakdown
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

type_counts = df['Conflict_Type'].value_counts()
colors = [type_colors.get(t, '#888') for t in type_counts.index]
axes[0].barh(type_counts.index, type_counts.values, color=colors, edgecolor='white')
axes[0].set_title('Conflict Type Distribution')
axes[0].set_xlabel('Count')
for i, v in enumerate(type_counts.values):
    axes[0].text(v + 5, i, str(v), va='center')

# Duration by conflict type
sns.boxplot(data=df, x='Conflict_Type', y='Duration_Days', ax=axes[1],
            palette=type_colors, order=type_counts.index)
axes[1].set_title('Duration by Conflict Type')
axes[1].set_xlabel('Conflict Type')
axes[1].set_ylabel('Duration (Days)')
plt.xticks(rotation=30, ha='right')

plt.tight_layout()
plt.show()

## 3. Country Analysis

In [ ]:
# Total involvement (as A or B)
all_countries = pd.concat([df['Country_A'], df['Country_B']]).value_counts()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].barh(all_countries.index[::-1], all_countries.values[::-1], color='steelblue', alpha=0.7)
axes[0].set_title('Total Conflict Involvement by Country')
axes[0].set_xlabel('Number of Conflicts')
for i, v in enumerate(all_countries.values[::-1]):
    axes[0].text(v + 3, i, str(v), va='center', fontsize=9)

# Country pairing heatmap
pairings = df.groupby(['Country_A', 'Country_B']).size().unstack(fill_value=0)
sns.heatmap(pairings, annot=True, fmt='d', cmap='YlOrRd', linewidths=0.5, ax=axes[1],
            annot_kws={'fontsize': 7})
axes[1].set_title('Conflict Pairings (Country A vs Country B)')
axes[1].tick_params(axis='both', labelsize=8)

plt.tight_layout()
plt.show()

## 4. Human Cost

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Total deaths over time
yearly_deaths = df.groupby('Year')[['Military_Deaths_A', 'Military_Deaths_B', 'Civilian_Deaths']].sum()
yearly_deaths.plot.area(ax=axes[0, 0], stacked=True, alpha=0.7,
                         color=['#3498DB', '#2980B9', '#E74C3C'])
axes[0, 0].set_title('Total Deaths by Year')
axes[0, 0].set_ylabel('Deaths')
axes[0, 0].legend(['Military A', 'Military B', 'Civilian'])

# Civilian vs Military deaths ratio
df['Civilian_Ratio'] = df['Civilian_Deaths'] / df['Total_Deaths']
yearly_ratio = df.groupby('Year')['Civilian_Ratio'].mean()
axes[0, 1].plot(yearly_ratio.index, yearly_ratio.values, color='#E74C3C', linewidth=1.5)
axes[0, 1].fill_between(yearly_ratio.index, yearly_ratio.values, alpha=0.2, color='#E74C3C')
axes[0, 1].set_title('Civilian Death Ratio Over Time')
axes[0, 1].set_ylabel('Civilian / Total Deaths')
axes[0, 1].set_xlabel('Year')

# Deaths by conflict type
death_by_type = df.groupby('Conflict_Type')[['Total_Military_Deaths', 'Civilian_Deaths']].mean()
death_by_type.plot(kind='bar', ax=axes[1, 0], color=['#3498DB', '#E74C3C'], alpha=0.7)
axes[1, 0].set_title('Average Deaths by Conflict Type')
axes[1, 0].set_ylabel('Average Deaths')
axes[1, 0].legend(['Military', 'Civilian'])
plt.sca(axes[1, 0])
plt.xticks(rotation=30, ha='right')

# Refugees over time
yearly_refugees = df.groupby('Year')['Refugees_Millions'].sum()
axes[1, 1].bar(yearly_refugees.index, yearly_refugees.values, color='#F39C12', alpha=0.7)
axes[1, 1].set_title('Total Refugees by Year (Millions)')
axes[1, 1].set_ylabel('Refugees (Millions)')
axes[1, 1].set_xlabel('Year')

plt.tight_layout()
plt.show()

print(f'Total deaths across all conflicts: {df["Total_Deaths"].sum():,.0f}')
print(f'Average civilian death ratio: {df["Civilian_Ratio"].mean():.1%}')
print(f'Total refugees: {df["Refugees_Millions"].sum():.1f} million')

## 5. Economic Impact

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Economic loss over time
yearly_econ = df.groupby('Year')['Economic_Loss_USD_Billions'].sum()
axes[0, 0].fill_between(yearly_econ.index, yearly_econ.values, alpha=0.4, color='#C0392B')
axes[0, 0].plot(yearly_econ.index, yearly_econ.values, color='#C0392B', linewidth=1.5)
axes[0, 0].set_title('Total Economic Loss by Year (USD Billions)')
axes[0, 0].set_ylabel('Loss (B USD)')

# Economic loss by conflict type
econ_type = df.groupby('Conflict_Type')['Economic_Loss_USD_Billions'].agg(['mean', 'sum'])
econ_type['mean'].sort_values().plot(kind='barh', ax=axes[0, 1], color='#E74C3C', alpha=0.7)
axes[0, 1].set_title('Average Economic Loss by Conflict Type')
axes[0, 1].set_xlabel('Avg Loss (B USD)')

# Economic loss vs Duration
axes[1, 0].scatter(df['Duration_Days'], df['Economic_Loss_USD_Billions'], 
                    alpha=0.2, s=10, c='#C0392B')
axes[1, 0].set_title('Economic Loss vs Duration')
axes[1, 0].set_xlabel('Duration (Days)')
axes[1, 0].set_ylabel('Loss (B USD)')

# GDP imbalance and outcome
df['GDP_Ratio'] = df['GDP_A_Billions'] / df['GDP_B_Billions']
for outcome, color in [('Victory_A', '#3498DB'), ('Stalemate', '#95A5A6'), ('Victory_B', '#E74C3C')]:
    subset = df[df['Outcome'] == outcome]
    axes[1, 1].hist(np.log10(subset['GDP_Ratio']), bins=30, alpha=0.5, color=color, label=outcome)
axes[1, 1].set_title('GDP Ratio (A/B) by Outcome (log scale)')
axes[1, 1].set_xlabel('log10(GDP_A / GDP_B)')
axes[1, 1].set_ylabel('Count')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

## 6. Geographic Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Scatter map of conflicts
scatter = axes[0].scatter(df['Longitude'], df['Latitude'], 
                           c=df['Total_Deaths'], cmap='YlOrRd', 
                           s=df['Economic_Loss_USD_Billions'] / 10, alpha=0.3)
axes[0].set_title('Conflict Locations (size=econ loss, color=deaths)')
axes[0].set_xlabel('Longitude')
axes[0].set_ylabel('Latitude')
plt.colorbar(scatter, ax=axes[0], label='Total Deaths')

# Terrain and Climate
terrain_climate = df.groupby(['Terrain_Type', 'Climate_Zone']).size().unstack(fill_value=0)
terrain_climate.plot(kind='bar', stacked=True, ax=axes[1], alpha=0.7,
                      colormap='Set2')
axes[1].set_title('Conflicts by Terrain & Climate')
axes[1].set_ylabel('Count')
axes[1].legend(title='Climate', bbox_to_anchor=(1.05, 1))
plt.sca(axes[1])
plt.xticks(rotation=0)

plt.tight_layout()
plt.show()

## 7. Military Factors

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Weapons used distribution
weapons = df['Weapons_Used'].value_counts()
weapon_colors = {'Nuclear': '#E74C3C', 'Cyber': '#3498DB', 'Conventional': '#2ECC71', 'Mixed': '#F39C12'}
axes[0, 0].pie(weapons.values, labels=weapons.index, autopct='%1.1f%%', startangle=90,
               colors=[weapon_colors.get(w, '#888') for w in weapons.index])
axes[0, 0].set_title('Weapons Used Distribution')

# Weapons over time
weapons_time = df.groupby(['Decade', 'Weapons_Used']).size().unstack(fill_value=0)
weapons_time.plot(kind='bar', stacked=True, ax=axes[0, 1], alpha=0.7,
                   color=[weapon_colors.get(w, '#888') for w in weapons_time.columns])
axes[0, 1].set_title('Weapons Used by Decade')
axes[0, 1].set_ylabel('Count')
axes[0, 1].legend(title='Weapon Type')
plt.sca(axes[0, 1])
plt.xticks(rotation=0)

# Air strikes distribution by conflict type
sns.boxplot(data=df, x='Conflict_Type', y='Air_Strikes', ax=axes[1, 0],
            palette=type_colors)
axes[1, 0].set_title('Air Strikes by Conflict Type')
axes[1, 0].set_ylabel('Air Strikes')
plt.sca(axes[1, 0])
plt.xticks(rotation=30, ha='right')

# Air strikes vs casualties
axes[1, 1].scatter(df['Air_Strikes'], df['Civilian_Deaths'], alpha=0.15, s=10, color='#E74C3C')
axes[1, 1].set_title('Air Strikes vs Civilian Deaths')
axes[1, 1].set_xlabel('Air Strikes')
axes[1, 1].set_ylabel('Civilian Deaths')

plt.tight_layout()
plt.show()

## 8. Diplomacy & Outcomes

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Outcome distribution
outcome_colors = {'Victory_A': '#3498DB', 'Stalemate': '#95A5A6', 'Victory_B': '#E74C3C'}
outcomes = df['Outcome'].value_counts()
axes[0, 0].bar(outcomes.index, outcomes.values, 
               color=[outcome_colors.get(o, '#888') for o in outcomes.index], alpha=0.7)
axes[0, 0].set_title('Conflict Outcomes')
axes[0, 0].set_ylabel('Count')
for i, v in enumerate(outcomes.values):
    axes[0, 0].text(i, v + 5, str(v), ha='center')

# Outcome by conflict type
outcome_type = pd.crosstab(df['Conflict_Type'], df['Outcome'], normalize='index') * 100
outcome_type[['Victory_A', 'Stalemate', 'Victory_B']].plot(
    kind='bar', stacked=True, ax=axes[0, 1], alpha=0.7,
    color=['#3498DB', '#95A5A6', '#E74C3C'])
axes[0, 1].set_title('Outcome Distribution by Conflict Type (%)')
axes[0, 1].set_ylabel('Percentage')
axes[0, 1].legend(title='Outcome')
plt.sca(axes[0, 1])
plt.xticks(rotation=30, ha='right')

# UN Involvement impact
un_outcomes = pd.crosstab(df['UN_Involvement'], df['Outcome'], normalize='index') * 100
un_outcomes.plot(kind='bar', ax=axes[1, 0], alpha=0.7,
                  color=['#95A5A6', '#3498DB', '#E74C3C'])
axes[1, 0].set_title('Outcomes by UN Involvement')
axes[1, 0].set_ylabel('Percentage')
axes[1, 0].legend(title='Outcome')
plt.sca(axes[1, 0])
plt.xticks(rotation=0)

# Sanctions and ceasefire impact on duration
sanction_groups = df.groupby(['Sanctions', 'Ceasefire'])['Duration_Days'].mean().unstack()
sanction_groups.plot(kind='bar', ax=axes[1, 1], alpha=0.7, color=['#E74C3C', '#2ECC71'])
axes[1, 1].set_title('Avg Duration by Sanctions & Ceasefire')
axes[1, 1].set_ylabel('Avg Duration (Days)')
axes[1, 1].set_xlabel('Sanctions')
axes[1, 1].legend(title='Ceasefire')
plt.sca(axes[1, 1])
plt.xticks(rotation=0)

plt.tight_layout()
plt.show()

## 9. Resource Disputes

In [ ]:
resource_df = df.dropna(subset=['Resource_Dispute'])

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Resource dispute frequency
res_counts = resource_df['Resource_Dispute'].value_counts()
res_colors = {'Oil': '#2C3E50', 'Land': '#8B4513', 'Water': '#3498DB'}
axes[0].bar(res_counts.index, res_counts.values, 
            color=[res_colors.get(r, '#888') for r in res_counts.index], alpha=0.7)
axes[0].set_title('Resource Dispute Types')
axes[0].set_ylabel('Count')

# Resource dispute by decade
res_decade = resource_df.groupby(['Decade', 'Resource_Dispute']).size().unstack(fill_value=0)
res_decade.plot(kind='bar', stacked=True, ax=axes[1], alpha=0.7,
                 color=[res_colors.get(r, '#888') for r in res_decade.columns])
axes[1].set_title('Resource Disputes by Decade')
axes[1].set_ylabel('Count')
axes[1].legend(title='Resource')
plt.sca(axes[1])
plt.xticks(rotation=0)

# Avg economic loss by resource
res_econ = resource_df.groupby('Resource_Dispute')['Economic_Loss_USD_Billions'].mean()
axes[2].bar(res_econ.index, res_econ.values, 
            color=[res_colors.get(r, '#888') for r in res_econ.index], alpha=0.7)
axes[2].set_title('Avg Economic Loss by Resource Type')
axes[2].set_ylabel('Avg Loss (B USD)')

plt.tight_layout()
plt.show()

no_resource = df['Resource_Dispute'].isna().sum()
print(f'Conflicts with resource dispute: {len(resource_df)} ({len(resource_df)/len(df)*100:.1f}%)')
print(f'Conflicts without resource dispute: {no_resource} ({no_resource/len(df)*100:.1f}%)')

## 10. Correlation Analysis

In [ ]:
corr_cols = ['Duration_Days', 'Military_Deaths_A', 'Military_Deaths_B', 'Civilian_Deaths',
             'Economic_Loss_USD_Billions', 'Air_Strikes', 'Naval_Battles', 'Refugees_Millions',
             'Population_A_Millions', 'GDP_A_Billions']
corr = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            square=True, linewidths=0.5, ax=ax)
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

## 11. Decade-Level Summary

In [ ]:
decade_summary = df.groupby('Decade').agg(
    Conflicts=('Year', 'count'),
    Avg_Duration=('Duration_Days', 'mean'),
    Total_Deaths=('Total_Deaths', 'sum'),
    Avg_Civilian_Ratio=('Civilian_Ratio', 'mean'),
    Total_Econ_Loss=('Economic_Loss_USD_Billions', 'sum'),
    Total_Refugees=('Refugees_Millions', 'sum'),
    Ceasefire_Rate=('Ceasefire', lambda x: (x == 'Yes').mean()),
    UN_Rate=('UN_Involvement', lambda x: (x == 'Yes').mean())
).round(2)

print(decade_summary.to_string())

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

axes[0, 0].bar(decade_summary.index, decade_summary['Total_Deaths'], color='#E74C3C', alpha=0.7, width=7)
axes[0, 0].set_title('Total Deaths by Decade')
axes[0, 0].set_ylabel('Deaths')

axes[0, 1].bar(decade_summary.index, decade_summary['Total_Econ_Loss'], color='#F39C12', alpha=0.7, width=7)
axes[0, 1].set_title('Total Economic Loss by Decade (B USD)')
axes[0, 1].set_ylabel('Loss (B USD)')

axes[1, 0].plot(decade_summary.index, decade_summary['Ceasefire_Rate'] * 100, 'o-', color='#2ECC71', linewidth=2)
axes[1, 0].set_title('Ceasefire Rate by Decade (%)')
axes[1, 0].set_ylabel('Ceasefire Rate (%)')
axes[1, 0].set_xlabel('Decade')

axes[1, 1].plot(decade_summary.index, decade_summary['UN_Rate'] * 100, 'o-', color='#3498DB', linewidth=2)
axes[1, 1].set_title('UN Involvement Rate by Decade (%)')
axes[1, 1].set_ylabel('UN Rate (%)')
axes[1, 1].set_xlabel('Decade')

plt.tight_layout()
plt.show()

## 12. Key Findings

Summary of the exploratory analysis of global conflict data (1950-2024):

- **Scale**: 3,000 conflicts across 15 countries over 75 years, with a roughly uniform distribution of ~40 conflicts per year
- **Conflict types**: Skirmishes are the most frequent (627), followed by Proxy Wars (621) and Civil Wars (597). Full-scale Wars are the least common (567)
- **Human cost**: Civilian deaths consistently outnumber military deaths across all conflict types. The average conflict produces ~100K civilian casualties and ~50K military casualties
- **Economic devastation**: Average economic loss per conflict is ~$254B USD. Economic loss shows weak correlation with conflict duration, suggesting intensity matters more than length
- **Country involvement**: All 15 countries are roughly equally involved, with the USA, Ukraine, and Russia among the most frequent participants
- **Weapons evolution**: Nuclear, Cyber, Conventional, and Mixed weapons are used in roughly equal proportions. Cyber warfare appears across all decades in this dataset
- **Diplomacy**: Outcomes are evenly split between Victory A, Victory B, and Stalemate (~33% each). UN involvement and sanctions show limited measurable impact on outcomes
- **Resource disputes**: Oil, Land, and Water disputes drive ~75% of conflicts with identified resource motives, split roughly equally
- **Geographic spread**: Conflicts are distributed globally across all terrain types and climate zones, with no dominant geographic cluster